# Monte Carlo Sample Visualization and Analysis

This notebook analyzes and visualizes Monte Carlo samples written by `cpp/main.exe`
(examples 3-5 in [main.cpp](../cpp/main.cpp)). Both velocity generators are now posed
in the **speed-squared** variable $w = |v|^2$ (rather than energy), so the particle
mass never enters: a user supplies a density $g(w)$, the generator draws $w$ by
rejection sampling and returns $|v| = \sqrt{w}$.

The two examples below were chosen so that every marginal has a closed form, which
lets each histogram be compared against the exact curve.

| Generator | Input $g(w)$ | Exact result |
|---|---|---|
| `general_velocity_generator` | $g(w) = \sqrt{w}\,e^{-w/\theta^2}$, $w=\lvert v\rvert^2$, isotropic direction | isotropic **Maxwellian**, $\theta = 1$: each component $\mathcal{N}(0, \theta^2/2)$, speed $\propto v^2 e^{-v^2/\theta^2}$ |
| `field_aligned_velocity_generator` | $g(w) = 1$ on $[0, 4]$, $w = v_\parallel^2$, $\theta_\perp = 1$, $\mathbf{B}\parallel(1,1,1)$ | **linear ramp** along $\mathbf{B}$: $p(v_\parallel) = v_\parallel/2$ on $[0,2]$, Maxwellian perpendicular plane |

The Jacobian $\mathrm{d}w = 2|v|\,\mathrm{d}|v|$ is what the plots mainly verify, and the
two rows show it from opposite directions: in the first, a $\sqrt{w}$ deliberately put
into $g$ comes back out as the $v^2$ of a true Maxwellian; in the second, a perfectly
flat $g$ still produces a sloped $v_\parallel$, because the Jacobian is there whether or
not you asked for it. The tilted $\mathbf{B}$ in the second row also exercises the
field-aligned rotation.

## Setup

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style for publication-quality figures
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['font.size'] = 10
plt.rcParams['axes.labelsize'] = 11
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['legend.fontsize'] = 9
plt.rcParams['figure.constrained_layout.use'] = True

# Generator parameters -- must match cpp/main.cpp
THETA_ISO  = 1.0            # isotropic Maxwellian thermal speed (example 3)
W_MAX_ISO  = 25.0           # truncation of w = |v|^2       (example 3)
THETA_PERP = 1.0            # perpendicular thermal speed    (example 5)
W_MAX_PAR  = 4.0            # upper edge of w = v_par^2      (example 5)
V_PAR_MAX  = np.sqrt(W_MAX_PAR)   # = 2, the hard cutoff in parallel speed
B_HAT      = np.array([1.0, 1.0, 1.0]) / np.sqrt(3.0)   # magnetic field direction

print("Libraries imported successfully.")

## Load Data into DataFrames

In [ ]:

# workspace_root = "./"
workspace_root = "../cpp/"

# Isotropic velocities from general_velocity_generator (example 3)
df_general_velocity = pd.read_csv(workspace_root + 'samples_general_velocity.txt', sep=r'\s+',
                                  names=['vx', 'vy', 'vz'], dtype=np.float64)

# Field-aligned velocities from field_aligned_velocity_generator (example 5), lab frame
df_field_aligned = pd.read_csv(workspace_root + 'samples_field_aligned.txt', sep=r'\s+',
                               names=['vx', 'vy', 'vz'], dtype=np.float64)

# 2D positions from general_position_generator (example 4). usecols keeps this working
# with the Python mirror in general_generators.py, which may pad the row out to 3D.
df_general_position = pd.read_csv(workspace_root + 'samples_general_position.txt', sep=r'\s+',
                                  header=None, usecols=[0, 1], names=['x', 'y'],
                                  dtype=np.float64)

print("Data loaded successfully.")
for name, df in [('General Velocity', df_general_velocity),
                 ('Field Aligned', df_field_aligned),
                 ('General Position', df_general_position)]:
    print(f"\n{name} Generator:\n  Shape: {df.shape}\n  First 3 rows:\n{df.head(3)}")

## Compute Derived Quantities

In [ ]:

# --- Isotropic set: speed, w = |v|^2, and the transverse magnitude -----------
def compute_velocity_quantities(df):
    df = df.copy()
    df['v_perp'] = np.hypot(df['vx'], df['vy'])                     # transverse magnitude
    df['speed'] = np.sqrt(df['vx']**2 + df['vy']**2 + df['vz']**2)  # |v|
    df['w'] = df['speed']**2                                        # w = |v|^2, the sampled variable
    return df

df_general_velocity = compute_velocity_quantities(df_general_velocity)

# --- Field-aligned set: project the lab-frame vectors onto the B frame -------
def field_aligned_frame(b_hat):
    """Right-handed orthonormal frame (e1, e2, e3) with e3 = B/|B|."""
    e3 = b_hat / np.linalg.norm(b_hat)
    seed = np.zeros(3)
    seed[np.argmin(np.abs(e3))] = 1.0    # least-aligned axis keeps the cross product stable
    e1 = np.cross(e3, seed)
    e1 /= np.linalg.norm(e1)
    e2 = np.cross(e3, e1)
    return e1, e2, e3

e1, e2, e3 = field_aligned_frame(B_HAT)
V_fa = df_field_aligned[['vx', 'vy', 'vz']].to_numpy()

df_field_aligned['v_par'] = V_fa @ e3
df_field_aligned['v_perp1'] = V_fa @ e1
df_field_aligned['v_perp2'] = V_fa @ e2
df_field_aligned['v_perp'] = np.hypot(df_field_aligned['v_perp1'], df_field_aligned['v_perp2'])
df_field_aligned['w_par'] = df_field_aligned['v_par']**2      # the sampled variable
df_field_aligned['phi_perp'] = np.arctan2(df_field_aligned['v_perp2'],
                                          df_field_aligned['v_perp1'])
df_field_aligned['speed'] = np.linalg.norm(V_fa, axis=1)

# --- Position set ------------------------------------------------------------
df_general_position['radius'] = np.hypot(df_general_position['x'], df_general_position['y'])

print("Derived quantities computed.")
print(f"  isotropic  : {len(df_general_velocity):,} samples")
print(f"  field-aligned: {len(df_field_aligned):,} samples "
      f"(frame e3 = {np.round(e3, 4)})")
print(f"  position   : {len(df_general_position):,} samples")

## 1. `general_velocity_generator` — isotropic Maxwellian from $g(w)=\sqrt{w}\,e^{-w/\theta^2}$

If the generator handles the change of variables $w = |v|^2$ correctly, the
$\sqrt{w}$ prefactor must reappear as the $v^2$ of the Maxwell speed distribution,
and every Cartesian component must be $\mathcal{N}(0,\theta^2/2)$. Dashed curves below
are those exact densities — no fitting.

In [ ]:

# 2D density maps: an isotropic sample must look circular in every plane.
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('General Velocity Generator: 2D Velocity Density Maps',
             fontsize=14, fontweight='bold', y=1.05)

pairs = [('vx', 'vy'), ('vx', 'vz'), ('vy', 'vz')]
for ax, (a, b) in zip(axes, pairs):
    hb = ax.hexbin(df_general_velocity[a], df_general_velocity[b],
                   gridsize=40, cmap='viridis', mincnt=1)
    ax.set_xlabel(f'$v_{a[1]}$')
    ax.set_ylabel(f'$v_{b[1]}$')
    ax.set_title(f'$v_{a[1]}$ vs $v_{b[1]}$')
    ax.set_aspect('equal')
    cb = plt.colorbar(hb, ax=ax)
    cb.set_label('Count', fontsize=10)

plt.show()

print("General velocity 2D density maps plotted.")

In [ ]:

# Histograms against the exact Maxwellian marginals (no fitted parameters).
sigma_iso = THETA_ISO / np.sqrt(2.0)     # per-component standard deviation

fig, axes = plt.subplots(2, 3, figsize=(14, 8), dpi=200)
fig.suptitle(r'General Velocity Generator, $g(w)=\sqrt{w}\,e^{-w/\theta^2}$ with '
             rf'$\theta={THETA_ISO:g}$: samples vs. theory',
             fontsize=14, fontweight='bold', y=1.05)


def hist_with_theory(ax, data, dist, label, xlabel, title, color, xlim=None):
    ax.hist(data, bins=120, density=True, color=color, alpha=0.7,
            edgecolor='black', linewidth=0.3, label='samples')
    lo, hi = (np.min(data), np.max(data)) if xlim is None else xlim
    grid = np.linspace(lo, hi, 400)
    ax.plot(grid, dist.pdf(grid), 'k--', linewidth=1.6, label=label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Probability density')
    ax.set_title(title)
    ax.grid(alpha=0.3)
    ax.legend()


norm_iso = stats.norm(0.0, sigma_iso)
for ax, comp in zip(axes[0], ['vx', 'vy', 'vz']):
    hist_with_theory(ax, df_general_velocity[comp], norm_iso,
                     rf'$\mathcal{{N}}(0,\theta^2/2)$', f'$v_{comp[1]}$',
                     f'$v_{comp[1]}$ distribution', 'steelblue')

# Transverse magnitude: Rayleigh with the same per-component sigma.
hist_with_theory(axes[1, 0], df_general_velocity['v_perp'],
                 stats.rayleigh(scale=sigma_iso),
                 'Rayleigh$(\\theta/\\sqrt{2})$', r'$v_{\perp}=\sqrt{v_x^2+v_y^2}$',
                 'Transverse speed', 'coral')

# Total speed: Maxwell -- this is the curve that tests the sqrt(w) Jacobian.
hist_with_theory(axes[1, 1], df_general_velocity['speed'],
                 stats.maxwell(scale=sigma_iso),
                 r'Maxwell $\propto v^2e^{-v^2/\theta^2}$', '$|v|$',
                 'Total speed', 'mediumseagreen')

# The sampled variable itself: w ~ Gamma(3/2, theta^2), i.e. g(w) normalised.
hist_with_theory(axes[1, 2], df_general_velocity['w'],
                 stats.gamma(1.5, scale=THETA_ISO**2),
                 r'$g(w)/\!\int\! g$', '$w=|v|^2$',
                 'Sampled variable $w$', 'mediumpurple', xlim=(0.0, W_MAX_ISO))

plt.show()

print("General velocity histograms plotted against exact Maxwellian marginals.")

## 2. `field_aligned_velocity_generator` — flat in $w_\parallel$, a ramp in $v_\parallel$

The simplest input there is: $g(w) = 1$ on $[0, 4]$, applied to $w = v_\parallel^2$ only.
A *constant* density in $w$ is **not** a constant density in $v_\parallel$ — the Jacobian
$\mathrm{d}w = 2v_\parallel\,\mathrm{d}v_\parallel$ tilts it into a straight line,

$$p(v_\parallel) = g(v_\parallel^2)\cdot 2v_\parallel = \frac{2v_\parallel}{w_{\max}}
= \frac{v_\parallel}{2}, \qquad 0 \le v_\parallel \le \sqrt{w_{\max}} = 2 .$$

That flat-becomes-a-ramp contrast is the cleanest picture of what the $w$ formulation
does, and it is worth comparing with §1, where the $\sqrt{w}$ put into $g$ came back out
as the $v^2$ of a Maxwellian. The two perpendicular components stay Maxwellian.

Because $\mathbf{B}\parallel(1,1,1)$ the lab-frame marginals are mixtures, so the samples
are first projected back onto the field-aligned frame
$(\mathbf{e}_1,\mathbf{e}_2,\mathbf{e}_3=\hat{\mathbf{B}})$ built above. Exact marginals
in that frame:

- $w_\parallel = v_\parallel^2 \sim$ Uniform$[0, 4]$ — i.e. $g$ itself
- $v_\parallel \sim$ the ramp $p = v_\parallel/2$ on $[0,2]$, strictly positive (`parallel_sign = +1`)
- $v_{\perp 1}, v_{\perp 2} \sim \mathcal{N}(0, \theta_\perp^2/2)$, and $v_\perp \sim$ Rayleigh$(\theta_\perp/\sqrt2)$
- the perpendicular azimuth is uniform on $[-\pi,\pi)$

In [ ]:

# Field-frame marginals against theory.
sigma_perp = THETA_PERP / np.sqrt(2.0)

fig, axes = plt.subplots(2, 3, figsize=(14, 8), dpi=200)
fig.suptitle(rf'Field-Aligned Velocity Generator, $g(w)=1$ on $[0,{W_MAX_PAR:g}]$ with '
             rf'$\theta_\perp={THETA_PERP:g}$, '
             r'$\mathbf{B}\parallel(1,1,1)$: samples vs. theory',
             fontsize=13, fontweight='bold', y=1.05)

# Parallel ramp: flat in w becomes a straight line in v_par.
hist_with_theory(axes[0, 0], df_field_aligned['v_par'],
                 stats.powerlaw(2.0, scale=V_PAR_MAX),
                 rf'ramp $2v_\parallel/{W_MAX_PAR:g}$', r'$v_{\parallel}$',
                 r'Parallel speed (linear ramp)', 'darkorange')
axes[0, 0].axvline(V_PAR_MAX, color='firebrick', linewidth=1.0, linestyle=':')

# Sampled variable: g(w_par) normalised is simply uniform.
hist_with_theory(axes[0, 1], df_field_aligned['w_par'],
                 stats.uniform(0.0, W_MAX_PAR),
                 r'$g(w)/\!\int\! g$ = uniform', r'$w_{\parallel}=v_{\parallel}^2$',
                 r'Sampled variable $w_{\parallel}$ (flat)', 'mediumpurple',
                 xlim=(0.0, W_MAX_PAR))
axes[0, 1].set_ylim(0.0, 0.4)

# Perpendicular magnitude.
hist_with_theory(axes[0, 2], df_field_aligned['v_perp'],
                 stats.rayleigh(scale=sigma_perp),
                 r'Rayleigh$(\theta_\perp/\sqrt{2})$', r'$v_{\perp}$',
                 'Perpendicular speed', 'coral')

# Both perpendicular components.
norm_perp = stats.norm(0.0, sigma_perp)
for ax, comp, lbl in [(axes[1, 0], 'v_perp1', r'$v_{\perp 1}$'),
                      (axes[1, 1], 'v_perp2', r'$v_{\perp 2}$')]:
    hist_with_theory(ax, df_field_aligned[comp], norm_perp,
                     r'$\mathcal{N}(0,\theta_\perp^2/2)$', lbl,
                     f'{lbl} distribution', 'steelblue')

# Perpendicular azimuth must be flat -- checks the rotation introduces no bias.
axes[1, 2].hist(df_field_aligned['phi_perp'], bins=120, density=True,
                color='mediumseagreen', alpha=0.7, edgecolor='black', linewidth=0.3,
                label='samples')
axes[1, 2].axhline(1.0 / (2.0 * np.pi), color='k', linestyle='--', linewidth=1.6,
                   label=r'uniform $1/2\pi$')
axes[1, 2].set_xlabel(r'$\phi_{\perp}=\mathrm{atan2}(v_{\perp 2},v_{\perp 1})$')
axes[1, 2].set_ylabel('Probability density')
axes[1, 2].set_title('Perpendicular azimuth')
axes[1, 2].set_ylim(0, 0.32)
axes[1, 2].grid(alpha=0.3)
axes[1, 2].legend()

plt.show()

print("Field-aligned marginals plotted against exact curves.")

In [ ]:

# Where the beam sits in velocity space: field frame (left) vs lab frame (right).
fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=200)
fig.suptitle('Field-Aligned Velocity Generator: beam geometry',
             fontsize=14, fontweight='bold', y=1.04)

hb = axes[0].hexbin(df_field_aligned['v_par'], df_field_aligned['v_perp1'],
                    gridsize=50, cmap='inferno', mincnt=1)
axes[0].set_xlabel(r'$v_{\parallel}$')
axes[0].set_ylabel(r'$v_{\perp 1}$')
axes[0].set_title(r'Field frame: one-sided, hard cutoff at $v_{\parallel}=2$')
axes[0].set_aspect('equal')
axes[0].axvline(0.0, color='white', linewidth=0.8, linestyle=':')
plt.colorbar(hb, ax=axes[0]).set_label('Count', fontsize=10)

hb = axes[1].hexbin(df_field_aligned['vx'], df_field_aligned['vz'],
                    gridsize=50, cmap='inferno', mincnt=1)
lim = 2.0
t = np.linspace(0.0, lim, 2)
axes[1].plot(t * B_HAT[0], t * B_HAT[2], color='cyan', linewidth=1.6,
             label=r'$\hat{\mathbf{B}}$ projected')
axes[1].set_xlabel('$v_x$')
axes[1].set_ylabel('$v_z$')
axes[1].set_title(r'Lab frame: tilted along $\mathbf{B}\parallel(1,1,1)$')
axes[1].set_aspect('equal')
axes[1].legend(loc='upper left')
plt.colorbar(hb, ax=axes[1]).set_label('Count', fontsize=10)

plt.show()

print("Beam geometry plotted.")

## 3. Quantitative validation

Moments and Kolmogorov–Smirnov distances against the exact distributions.

Ten marginals are tested at once, so a plain 5% per-test threshold would be expected to
flag about one of them by chance even if every generator were perfect. The threshold
below is therefore Bonferroni-corrected to a **family-wise** 5%: each test is judged at
$\alpha/10$, using the asymptotic KS critical value $c(\alpha)/\sqrt{N}$ with
$c(\alpha)=\sqrt{-\tfrac12\ln(\alpha/2)}$. The uncorrected per-test $p$ is reported in
the same table, so nothing is hidden by the correction.

In [ ]:

checks = [
    # (dataset label, quantity, samples, exact distribution)
    ('isotropic', 'v_x',           df_general_velocity['vx'],     stats.norm(0.0, sigma_iso)),
    ('isotropic', 'v_y',           df_general_velocity['vy'],     stats.norm(0.0, sigma_iso)),
    ('isotropic', 'v_z',           df_general_velocity['vz'],     stats.norm(0.0, sigma_iso)),
    ('isotropic', '|v|',           df_general_velocity['speed'],  stats.maxwell(scale=sigma_iso)),
    ('isotropic', 'w = |v|^2',     df_general_velocity['w'],      stats.gamma(1.5, scale=THETA_ISO**2)),
    ('field-aligned', 'v_par',     df_field_aligned['v_par'],     stats.powerlaw(2.0, scale=V_PAR_MAX)),
    ('field-aligned', 'w_par',     df_field_aligned['w_par'],     stats.uniform(0.0, W_MAX_PAR)),
    ('field-aligned', 'v_perp1',   df_field_aligned['v_perp1'],   stats.norm(0.0, sigma_perp)),
    ('field-aligned', 'v_perp2',   df_field_aligned['v_perp2'],   stats.norm(0.0, sigma_perp)),
    ('field-aligned', 'v_perp',    df_field_aligned['v_perp'],    stats.rayleigh(scale=sigma_perp)),
]

ALPHA = 0.05
alpha_test = ALPHA / len(checks)                      # Bonferroni: family-wise 5%
c_alpha = np.sqrt(-0.5 * np.log(alpha_test / 2.0))    # asymptotic two-sided KS constant

rows = []
for dataset, name, data, dist in checks:
    data = np.asarray(data)
    ks = stats.kstest(data, dist.cdf)
    crit = c_alpha / np.sqrt(data.size)
    rows.append({
        'dataset': dataset,
        'quantity': name,
        'mean': data.mean(),
        'mean (exact)': dist.mean(),
        'std': data.std(ddof=1),
        'std (exact)': dist.std(),
        'KS': ks.statistic,
        'KS crit': crit,
        'p': ks.pvalue,
        'verdict': 'PASS' if ks.statistic < crit else 'FAIL',
    })

summary = pd.DataFrame(rows)
display(summary.style.format({
    'mean': '{:.4f}', 'mean (exact)': '{:.4f}',
    'std': '{:.4f}', 'std (exact)': '{:.4f}',
    'KS': '{:.5f}', 'KS crit': '{:.5f}', 'p': '{:.3f}',
}).hide(axis='index'))

n_fail = int((summary['verdict'] == 'FAIL').sum())
print(f"\n{len(summary) - n_fail}/{len(summary)} marginals agree with theory "
      f"at a family-wise {ALPHA:.0%} level (per-test alpha = {alpha_test:.4f}, "
      f"KS threshold {c_alpha:.3f}/sqrt(N)).")
print(f"smallest per-test p: {summary['p'].min():.4f} "
      f"({summary.loc[summary['p'].idxmin(), 'quantity']}) -- with {len(summary)} tests, "
      f"a p this small is expected {1 - (1 - summary['p'].min())**len(summary):.0%} of the "
      f"time even for a perfect sampler.")

# Isotropy of the general_velocity_generator: the three component variances and the
# mean direction should be indistinguishable.
V_iso = df_general_velocity[['vx', 'vy', 'vz']].to_numpy()
print("\nIsotropy check (general_velocity_generator):")
print(f"  component variances : {np.round(V_iso.var(axis=0), 5)}  (exact {THETA_ISO**2 / 2:.5f})")
print(f"  mean direction      : {np.round(V_iso.mean(axis=0), 5)}  (exact [0 0 0])")

# Alignment check: the mean field-aligned velocity must point along B.
mean_fa = df_field_aligned[['vx', 'vy', 'vz']].to_numpy().mean(axis=0)
cos_align = mean_fa @ B_HAT / np.linalg.norm(mean_fa)
print("\nAlignment check (field_aligned_velocity_generator):")
print(f"  <v>                 : {np.round(mean_fa, 5)}")
print(f"  cos angle to B      : {cos_align:.6f}  (exact 1)")
print(f"  <v_par>             : {df_field_aligned['v_par'].mean():.5f}  "
      f"(exact {2.0 * V_PAR_MAX / 3.0:.5f})")
print(f"  max v_par           : {df_field_aligned['v_par'].max():.5f}  "
      f"(exact cutoff {V_PAR_MAX:.5f})")

## 4. `general_position_generator` — $\rho(x,y) = 1 + \sin(x)\sin(y)$

In [ ]:

# General position generator: Histograms of x, y, and radius (2D)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('General Position Generator ($\\rho(x,y) = 1 + \\sin(x)\\sin(y)$): 2D Analysis',
             fontsize=14, fontweight='bold', y=1.05)

axes[0].hist(df_general_position['x'], bins=100, color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.5)
axes[0].set_xlabel('$x$')
axes[0].set_ylabel('Count')
axes[0].set_title('$x$ Distribution')
axes[0].grid(alpha=0.3)

axes[1].hist(df_general_position['y'], bins=100, color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.5)
axes[1].set_xlabel('$y$')
axes[1].set_ylabel('Count')
axes[1].set_title('$y$ Distribution')
axes[1].grid(alpha=0.3)

axes[2].hist(df_general_position['radius'], bins=100, color='darkorange', alpha=0.7, edgecolor='black', linewidth=0.5)
axes[2].set_xlabel('$r$')
axes[2].set_ylabel('Count')
axes[2].set_title('Radial Distance Distribution')
axes[2].grid(alpha=0.3)

plt.show()

print("General 2D position histograms plotted.")

In [ ]:

# General position generator: sampled density next to the reference function.
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=200)
fig.suptitle('General Position Generator: samples vs. reference density',
             fontsize=14, fontweight='bold', y=1.02)

hb = axes[0].hexbin(df_general_position['x'], df_general_position['y'],
                    gridsize=45, cmap='plasma', mincnt=0)
axes[0].set_xlim(-np.pi, np.pi)
axes[0].set_ylim(-np.pi, np.pi)
axes[0].set_xlabel('$x$')
axes[0].set_ylabel('$y$')
axes[0].set_title('$x$ vs $y$ sampled density')
axes[0].set_aspect('equal')
plt.colorbar(hb, ax=axes[0]).set_label('Count', fontsize=10)

n = 400
grid = np.linspace(-np.pi, np.pi, n)
X, Y = np.meshgrid(grid, grid)
Z = 1 + np.sin(X) * np.sin(Y)
im = axes[1].imshow(Z, extent=(-np.pi, np.pi, -np.pi, np.pi), origin='lower',
                    aspect='equal', cmap='plasma')
axes[1].set_xlabel('$x$')
axes[1].set_ylabel('$y$')
axes[1].set_title(r'Reference $\rho(x,y)=1+\sin(x)\sin(y)$')
plt.colorbar(im, ax=axes[1]).set_label(r'$\rho(x,y)$', fontsize=10)

plt.show()

print("General 2D position map plotted alongside the reference density.")